Below we paste the code in Prolog and Python developed for the assignment. 

While this code is not executable in a Jupyter Notebook environment below, we have successfully executed it via Forum Code Notebooks.

Original Forum Code Link: https://sle-collaboration.minervaproject.com/?id=305bc5da-e60c-4584-9fe1-0440f088d55e&userId=11835&name=Uchechukwu+Unanka&avatar=https%3A//s3.us-east-1.amazonaws.com/picasso.fixtures/Uchechukwu_Unanka_11835_2022-12-28T03%3A12%3A33.269Z&isInstructor=0&signature=c8287fe12f16f12cbec7ef234ec76dd099368f2f9875c80caea9fcbce4fc0cf1.

In [ ]:
#!pip install nltk
#import nltk
#nltk.download('stopwords')
!pip install pylcs
try:
    import pyswip
except ImportError:
    !pip install pyswip
    import pyswip

In [ ]:
KB = """
%  Tell prolog that known/3 and multivalued/1 will be added later
:- dynamic known/3, multivalued/1.

%/ answers for shoppings
place(yongkang_street):- location(taipei), intention(shopping), buy(souvenirs), price(high).

place(taipei101) :-  location(taipei), intention(shopping), (buy(food); buy(clothes)), price(high).

place(gongguan_market) :-  location(taipei), intention(shopping), (buy(souvenirs); buy(clothes)),price(low).

place(raohe_market) :- location(taipei), intention(shopping), buy(food), price(low).

place(zhongshan_street) :- location(taipei), intention(shopping), buy(clothes), price(medium).

place(dihua_street) :- location(taipei), intention(shopping), (buy(souvenirs); buy(food)), price(medium).



%answers for learning
place(taipei_fine_art_museum) :- location(taipei), intention(learning), entrance_fee(yes), (about(culture); about(history)).

place(national_palace_museum) :- location(taipei), intention(learning),  entrance_fee(yes), (about(culture); about(history)).

place(chiang_shek_memorial_hall) :- location(taipei), intention(learning), entrance_fee(no), (about(culture); about(history)).

place(huashan_creative_park) :- location(taipei), intention(learning),  entrance_fee(no), (about(art); about(history)).



%answers for relaxing and destress
place(eslite_bookstore_taipei101) :- location(taipei), intention(relaxing),  crowded(yes),setting(indoor), (ambience(cozy); ambience(tranquil)).

place(elephant_mountain) :- location(taipei), intention(relaxing), crowded(no), setting(outdoor), ambience(tranquil).

place(tam_sui_river) :- location(taipei), intention(relaxing), crowded(yes), setting(outdoor), ambience(tranquil).

place(daan_park) :- location(taipei), intention(relaxing), crowded(no), setting(outdoor), ambience(cozy).

place(songshan_creative_park) :- location(taipei), intention(relaxing), crowded(no), (setting(indoor); setting(outdoor)), ambience(lively).

place(jiufen) :- location(taipei), intention(relaxing), crowded(yes), (setting(indoor); setting(outdoor)), ambience(lively).




%This checks if the user is not in taipei
place(ask_others) :- \+location(taipei).




% The code below implements the prompting to ask the user:

% Define the buy rule
buy(X) :- menuask('What would you like to buy? Input text or select options', X, [souvenirs, clothes, food]).

% Define the intention rule
intention(X) :- menuask('What is your intention of going to a tourist place? Input text or select options', X, [relaxing, learning, shopping]).

% Define the price rule
price(X) :- menuask('What is your preferred price range?'Input text or select options, X, [low, medium, high]).

% Define the location rule
location(X) :- ask(location, X).

% Define the entrance_fee rule
entrance_fee(X) :- menuask('Do you mind paying an entrance fee?Input text or select options', X, [yes, no]).

% Define the about rule
about(X) :- menuask('What do you want to learn about? Input text or select options', X, [art, history, culture]).

% Define the crowded rule
crowded(X) :- menuask('Do you prefer a crowded setting? Input text or select options', X, [yes, no]).

% Define the setting rule
setting(X) :- menuask('What type of settings do you want? Input text or select options', X, [indoor, outdoor]).

% Define the ambience rule
ambience(X) :- menuask('What type of ambience do you want? Input text or select options', X, [lively, cozy, tranquil]).


% Asking clauses
ask(A, V):-
known(yes, A, V), % succeed if true
!.	% stop looking

ask(A, V):-
known(_, A, V), % fail if false
!, fail.


% If not multivalued, and already known, don't ask again for a different value.
ask(A, V):-
\+multivalued(A),
known(yes, A, V2),
V \== V2,
!.


ask(A, V):-
read_py(A,V,Y), % get the answer
assertz(known(Y, A, V)), % remember it
Y == yes.	% succeed or fail

%Reference: http://www.amzi.com/ExpertSystemsInProlog/02usingprolog.php

menuask(A, V, _):-
known(yes, A, V), % succeed if true
!.	% stop looking

menuask(A, V, _):-
known(yes, A, V2), % If already known, don't ask again for a different value
V \== V2,
!, fail.

menuask(A, V, Menu):-
  read_menu_py(A,X,Menu),
  confirm_answer(X,A,V,Menu),
  asserta(known(yes,A,X)),
  X == V.


confirm_answer(X,_,_,Menu):-
  member(X,Menu),
  !.

confirm_answer(X,A,V,Menu):-
 dialog_response(X), dialog_response(' Please, change your input.\n'),
 menuask(A, V, Menu). 

"""

In [ ]:
# The code here will ask the user for input based on the askables. It will only ask the user where necessary.

# Import necessary packages
import tempfile
import numpy as np
from pyswip.prolog import Prolog
from pyswip.easy import *
import re
import pylcs
#import nltk 


prolog = Prolog() # Global handle to interpreter

retractall = Functor("retractall")
known = Functor("known",3)

responses = {'yongkang_street': "Yongkang street: https://goo.gl/maps/FuvJmag1fDuon3yZ7 ",
             'taipei101': "Taipei 101: https://goo.gl/maps/5uFYd1T1DnWyeinj7 ",
             'raohe_market': "Raohe Market: https://goo.gl/maps/HxueeHweQKqSpQer6 ",
             'zhongshan_street': "Zhongshan Street: https://goo.gl/maps/pBAUduvuDccmwkLv9 ",
             'dihua_street': "Dihua Street: https://goo.gl/maps/RkEwL6picyg4U5Wf9 ",
             'taipei_fine_art_museum': "Taipei Fine Arts Museum: https://goo.gl/maps/ZoCKihpggAxrGTSW9 ",
             'national_palace_museum': "National Palace Museum: https://goo.gl/maps/fqf11Kq3rcDGJfuB9 ",
             'chiang_shek_memorial_hall': "Chiang Kai-Shek Memorial Hall: https://goo.gl/maps/AnfcieZFnvhpiScZ8 ",
             'huashan_creative_park': "Huashan 1914 Creative Park: https://goo.gl/maps/psRLb7XZYHxbZkzT9 ",
             'eslite_bookstore_taipei101': "Eslite Bookstore Taipei101: https://goo.gl/maps/RHD2NT5b3ZwZnuVH7 ",
             'elephant_mountain': "Elephant Mountain: https://goo.gl/maps/VQXRm5UZMkHEtrv99 ",
             'daan_park': "Daan Forest Park: https://goo.gl/maps/tVKErFBuhqdwfQU49 ",
             'songshan_creative_park': "Songshan Creative Park: https://goo.gl/maps/Z3EgqF8NRLYGAaW28 ",
             'jiufen': "Jiufen: https://goo.gl/maps/9ibSr6LszsTN6BsR8 ",
             'gongguan_market': "Gongguan Market: https://goo.gl/maps/LriJznxCXsJCqnPE8 ",
             'tam_sui_river':"Tamsui River: https://goo.gl/maps/R4MouMMvtWzWmgWS8 "
            }

# Define foreign functions for getting user input and writing to the screen
def write_py(X):
    print(str(X))
    sys.stdout.flush()
    return True

def read_py(A,V,Y):
    if isinstance(Y, Variable):
        response = input("Confirm that your " + str(A) + " is " + str(V).capitalize() + "? (type yes/no)")
        Y.unify(response)
        return True
    else:
        return False


def classify_input(response_input, Menu):
    try:
        # if users input an integer
        if int(response_input) in range(1, len(Menu) + 1):
            return int(response_input)
        else:
            return False
    except ValueError:
        # if users input a non-integer string
        response = find_answer(Menu, response_input)
        if len(response_input) ==0:
            return False
        return response + 1
    return False
           
def read_menu_py(A, Y, Menu):
    Menu = [atom.value for atom in Menu]
    if isinstance(Y, Variable):
        menu_questions = "" + str(A) + "\n"
        for i in range(len(Menu)):
            menu_questions += f"{i+1}. {str(Menu[i])} \n"
        print(menu_questions)
        while True:
            try:
                response_input = input("Previous input: ".format(len(Menu)))
                response_input = classify_input(response_input,Menu) 
                #print(response_input)
                if response_input is False:
                    raise ValueError
                break
            except ValueError:
                print("Invalid input. Please try again.\n")
        response = str(Menu[int(response_input) - 1])
        Y.unify(response)
        return True
    else:
        return False


#Reference: https://www.geeksforgeeks.org/normalizing-textual-data-with-python/
def normalize_text(text):
    # ensure its lowercase
    text = text.lower()
    # remove numbers
    text = re.sub(r'\d+', '', text)
    # remove punctuations
    text = re.sub(r'[^\w\s]','',text) 
    # remove 's from the end of words
    text = re.sub(r'\b(\w+)(\'s)\b', r'\1', text)  
    # remove spaces
    text = re.sub(r'\s+', ' ', text)
    # remove stopwords 
    #stopwords = set(nltk.corpus.stopwords.words('english'))
    #text = ''.join([word for word in text.split() if word not in stopwords])
    text = ''.join([word for word in text.split()])
    # return the normalized text
    return text.strip()
    
def find_answer(list_options, response):
    """
    Using lcs to match patterns"""
    #normalize data
    list_texts = []
    length = []
    for i in list_options:
        text_normalized = normalize_text(i)
        list_texts.append(text_normalized)
    response = normalize_text(response)
    lcs_list = pylcs.lcs_of_list(response, list_options)
    for option in list_texts:
        length.append(len(option))
    similar_rate = np.array(lcs_list)/np.array(length)
    # find option with highest similarity rate
    answer_idx= np.array(similar_rate).argmax()
    
    #if the similarity rate between the possible 
    #response and the user response is less than 0.2
    #it is likely to be a coincidence
    if similar_rate[answer_idx] < 0.2:
        return False
    return int(answer_idx)
    
write_py.arity = 1
read_py.arity = 3
read_menu_py.arity = 3


registerForeign(read_py)
registerForeign(read_menu_py)
registerForeign(write_py)

# Create a temporary file with the KB in it
(FD, name) = tempfile.mkstemp(suffix='.pl', text = "True")
with os.fdopen(FD, "w") as text_file:
    text_file.write(KB)
prolog.consult(name) # open the KB for consulting
os.unlink(name) # Remove the temporary file

call(retractall(known))
place = [s for s in prolog.query("place(X).", maxresult=1)]

if place and place[0]['X'] != 'ask_others':
    print("Your recommendation is " + responses[place[0]['X']] + ".")
elif place and place[0]['X'] == 'ask_others':
    print("Sorry, we don't have any recommendations based on your preferences, but you can ask others for suggestions.")
else:
    print("Sorry, we don't have any recommendations based on your preferences, but you can ask others for suggestions.")